# 01 - Getting started with Delphos

This notebook is the shortest path from a trained Delphos checkpoint to candidate choice-model specifications.

You will learn how to:

- list bundled datasets
- load the final-user Delphos agent
- propose models without running Apollo/R
- inspect terms, parameters, and generated Apollo code
- save a proposal table for later review


## 1. Import Delphos


In [1]:
import delphos as dp

print("Delphos is ready")


Delphos is ready


## 2. See bundled datasets

The package ships with example datasets that are already mapped to the trained Delphos catalogue.


In [2]:
datasets = dp.list_datasets()
for item in datasets:
    print(f"{item.id:>2} | {item.name:<24} | {item.folder}")


 1 | ApolloModeChoice         | dataset_1
 2 | SwissmetroRouteChoice    | dataset_2
 3 | Decisions                | dataset_3
 4 | Swissmetro               | dataset_4
 5 | NLModeChoice             | dataset_5
 6 | NorwayVTT                | dataset_6
 7 | Arentze2013              | dataset_7
 8 | SpainParkingchoice       | dataset_8
 9 | LondonModeChoice         | dataset_9
10 | Optima                   | dataset_10
11 | VanCranenburghVOT        | dataset_11


## 3. Load one dataset and the trained agent

The default checkpoint is the production multitask checkpoint included in `checkpoints/full_agent_task_10_seed_123`.


In [3]:
dataset = dp.load_dataset("Swissmetro")
agent = dp.load_agent()

print(dataset)
print(agent.agent.summary())


Task(name='Swissmetro', alternatives=3, attributes=5, covariates=11)
{'agent': 'DelphosAgent', 'mode': 'inference', 'encoder_kind': 'deepset', 'state_dim': 64, 'num_actions': 297, 'z_cfg': {'K': 7, 'T': 3, 'G': 2, 'C': 7, 'd_att': 16, 'd_tr': 8, 'd_taste': 8, 'd_cov': 16, 'd_term': 64, 'd_state': 128, 'context_dim': 0, 'head_flag': False, 'pooling': 'mean', 'attention_heads': 4, 'attention_layers': 1, 'attention_dropout': 0.0}, 'device': 'cpu'}


## 4. Propose models without estimation

`estimate=False` is the default. This is fast because Delphos only searches the modelling space and builds Apollo-ready specifications. It does not call R.


In [4]:
models = agent.propose(
    dataset,
    n_models=5,
    max_attempts=50,
    strategy="topk",
    top_k=5,
    temperature=0.8,
    seed=123,
)

models.to_dataframe()


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices
0,4,Swissmetro,1110_2120_3212_4324_5000_6110_7000,10,topk,0,False,None,5,"[125, 74, 215, 106, 201, 75, 21, 17, 125, 149]"
1,4,Swissmetro,1110_2120_3210_4111_5000_6126_7000,10,topk,1,False,None,5,"[215, 125, 21, 17, 201, 27, 215, 73, 17, 106]"
2,4,Swissmetro,1110_2120_3210_4214_5000_6110_7000,10,topk,2,False,None,5,"[21, 125, 74, 75, 73, 215, 17, 75, 201, 73]"
3,4,Swissmetro,1110_2124_3210_4214_5000_6110_7000,10,topk,3,False,None,5,"[73, 27, 74, 215, 21, 201, 73, 215, 125, 201]"
4,4,Swissmetro,1110_2120_3212_4324_5000_6126_7000,10,topk,4,False,None,5,"[17, 27, 73, 21, 215, 106, 27, 75, 17, 149]"


## 5. Inspect the first proposal

Each proposal has a stable specification key, the active Delphos terms, the action sequence, and an Apollo specification object.


In [5]:
proposal = models.proposals[0]

print("Specification key:", proposal.specification_key)
print("Episode length:", proposal.episode_length)
print("Action indices:", proposal.action_indices)
print("Terms:")
for term in proposal.terms:
    print(term)


Specification key: 1110_2120_3212_4324_5000_6110_7000
Episode length: 10
Action indices: [125, 74, 215, 106, 201, 75, 21, 17, 125, 149]
Terms:
Term(attribute_id=1, transform_id=1, taste_id=1, covariate_id=0)
Term(attribute_id=2, transform_id=1, taste_id=2, covariate_id=0)
Term(attribute_id=3, transform_id=2, taste_id=1, covariate_id=2)
Term(attribute_id=4, transform_id=3, taste_id=2, covariate_id=4)
Term(attribute_id=6, transform_id=1, taste_id=1, covariate_id=0)


## 6. Inspect generated Apollo components

These objects are what Delphos sends to the environment when `estimate=True`.


In [8]:
apollo_spec = proposal.apollo_specification

print("Number of parameters:", apollo_spec.n_parameters)
print("First parameters:", apollo_spec.parameter_names[:10])
print()
print("Utility code preview:")
print()
print(apollo_spec.utility_code)


Number of parameters: 16
First parameters: ['ASC_TRAIN', 'ASC_SM', 'ASC_CAR', 'b_TRAIN_time', 'b_SM_time', 'b_CAR_time', 'b_cost_generic_log_income_1', 'b_cost_generic_log_income_2', 'b_cost_generic_log_income_3', 'b_cost_generic_log_income_4']

Utility code preview:

V <- list()

V[["TRAIN"]] <-
      ASC_TRAIN +
      b_TRAIN_time * train_tt_scaled +
      b_cost_generic_log_income_1 * (income == 1) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_2 * (income == 2) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_3 * (income == 3) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_4 * (income == 4) * log(1+train_cost_scaled) +
      b_TRAIN_headway_box_cox_purpose_1 * (purpose == 1) * ((train_he_scaled^L_headway - 1) / L_headway) +
      b_TRAIN_headway_box_cox_purpose_2 * (purpose == 2) * ((train_he_scaled^L_headway - 1) / L_headway)

V[["SM"]] <-
      ASC_SM +
      b_SM_time * sm_tt_scaled +
      b_cost_generic_log_income_1 * (income == 1) * log

## 7. Save proposals

This is a useful pattern when you want to inspect models first, then estimate them later.


In [10]:
output_path = "getting_started_proposals.csv"
models.to_dataframe().to_csv(output_path, index=False)
print(f"Saved {len(models)} proposals to {output_path}")


Saved 5 proposals to getting_started_proposals.csv


## 8. Optional: estimate through Apollo/R

Set `RUN_ESTIMATION = True` when R and Apollo are installed and you are ready to estimate. The environment creates a local SQLite cache automatically.


In [11]:
RUN_ESTIMATION = True

if RUN_ESTIMATION:
    estimated = agent.propose(
        dataset,
        n_models=1,
        strategy="greedy",
        estimate=True,
        estimate_kwargs={"max_free_parameters": 30},
    )
    display(estimated.to_dataframe())
else:
    print("Skipping Apollo/R estimation. Set RUN_ESTIMATION = True to run it.")


Apollo ignition sequence completed
Several observations per individual detected based on the value of id.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices,...,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,4,Swissmetro,1110_2212_3210_4214_5000_6110_7000,10,greedy,0,True,0.082363,5,"[215, 17, 201, 125, 21, 215, 17, 27, 201, 73]",...,0.267478,0.266042,0.130133,0.128769,10223.540776,10291.740385,-9.897254,0.739651,10,0
